# Encoder Comparison for Biomedical Retrieval
## A factorial evaluation on a gold-standard query set

**Objective:** This notebook evaluates the relative contribution of domain specificity and retrieval fine-tuning to semantic retrieval quality in a biomedical RAG pipeline. The LLM generation layer is not being evaluated.

**Metrics:**
- **Precision@5** — of the top 5 retrieved abstracts, how many are relevant?
- **Recall@5** — of all known relevant abstracts, how many did the encoder find in the top 5?
- **MRR (Mean Reciprocal Rank)** — at what position did the first relevant abstract appear?

**Experimental design:**

Three encoders were selected to isolate the contribution of each factor independently:

| Encoder | Domain-specific | Retrieval fine-tuned |
|---|---|---|
| `all-MiniLM-L6-v2` | No | Yes (MS-MARCO) |
| `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract` | Yes (PubMed) | No |
| `pritamdeka/S-PubMedBert-MS-MARCO` | Yes (PubMed) | Yes (MS-MARCO) |

This factorial structure allows direct attribution of performance differences: MiniLM vs PubMedBERT isolates the effect of domain specificity; PubMedBERT vs S-PubMedBERT isolates the effect of retrieval fine-tuning on a domain-specific base model; MiniLM vs S-PubMedBERT captures the combined effect.

**Gold-standard dataset:** 20 biomedical queries spanning oncology, immunology, neuroscience, liver biology, metabolic disease, and cell biology. For each query, 3-5 original research articles and 1-2 review articles were identified manually by the author, a domain expert (PhD, biomedical sciences). Two evaluations are reported: original research PMIDs only (harder, more discriminating) and combined research + reviews (more lenient).

**Corpus construction:** the top 60 PubMed abstracts per query were fetched using keyword search (Best Match ranking) and pooled across all queries, yielding ~1047 unique abstracts. Gold-standard abstracts absent from keyword search results were added explicitly by PMID fetch. Final gold-standard coverage: 103/104 PMIDs present in corpus.

## 1. Setup

In [1]:
!pip install requests sentence-transformers chromadb python-dotenv -q

In [2]:
import requests
import time
import re
import os
import json
import numpy as np
import pandas as pd
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from utils import (
    fetch_pubmed_abstracts_batched,
    clean_abstract,
    parse_abstracts_with_metadata,
    fetch_structured_metadata,
)

load_dotenv()
NCBI_API_KEY = os.getenv("NCBI_API_KEY")
print(f"NCBI key loaded: {NCBI_API_KEY is not None}")

2026-06-29 17:05:48.797008: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 17:05:48.798105: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-29 17:05:48.802953: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-29 17:05:48.816117: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-29 17:05:48.842377: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

NCBI key loaded: True


## 2. Gold-Standard Dataset

20 queries across oncology, immunology, neuroscience, liver disease, metabolic disease, and cell biology.

For each query:
- `query` — natural language question passed to the encoder at retrieval time
- `pubmed_search` — keyword string used to fetch the corpus from PubMed (Best Match ranking)
- `relevant_pmids` — 3–5 PMIDs from **original research articles** identified manually as directly relevant by a domain expert
-  `relevant_pmids_reviews` — 1–2 PMIDs from **review articles** identified manually as directly relevant by a domain expert


**Relevance criterion:** an abstract is relevant if it directly addresses the main subject of the query. Abstracts mentioning the topic only in passing are excluded.

In [3]:
GOLD_STANDARD = {
    "Q01": {
        "query": "What is the mechanistic link between BRCA1 and DNA repair in breast cancer?",
        "pubmed_search": "BRCA1 DNA repair mechanism breast cancer",
        "relevant_pmids": [29635390, 31251913, 39172790],
        "relevant_pmids_reviews": [32094664, 39261728]
    },
    "Q02": {
        "query": "Which genes are implicated in idiopathic pulmonary fibrosis?",
        "pubmed_search": "idiopathic pulmonary fibrosis genetic risk factors genes",
        "relevant_pmids": [36602845,31710517, 39838395],
        "relevant_pmids_reviews": [38573068, 23638821]
    },
    "Q03": {
        "query": "What are the biomarkers of NASH progression to cirrhosis?",
        "pubmed_search": "NASH biomarkers fibrosis progression cirrhosis",
        "relevant_pmids": [35226336, 32610095, 34105780, 40181314],
        "relevant_pmids_reviews": [30660725]
    },
    "Q04": {
        "query": "What mechanisms underlie resistance to anti-PD1 immunotherapy?",
        "pubmed_search": "anti-PD1 immunotherapy resistance mechanisms tumour",
        "relevant_pmids": [36932387, 35219791, 34753486, 34838121, 32275969],
        "relevant_pmids_reviews": [32793604]
    },
    "Q05": {
        "query": "What is the role of alpha-synuclein aggregation in Parkinson's disease?",
        "pubmed_search": "alpha-synuclein aggregation Parkinson's disease pathology",
        "relevant_pmids": [40314842, 40756347, 41538327],
        "relevant_pmids_reviews": [40140103, 11215516]
    },
    "Q06": {
        "query": "What drives CAR-T cells to become exhausted?",
        "pubmed_search": "CAR-T exhaustion mechanism",
        "relevant_pmids": [39333315, 34788079, 39438476 ],
        "relevant_pmids_reviews": [35301179, 38582965]
    },
    "Q07": {
        "query": "What is the mechanism of action of GIP receptor agonists in type 2 diabetes?",
        "pubmed_search": "GIP receptor agonist mechanism type 2 diabetes",
        "relevant_pmids": [38878772, 34003802, 37946085],
        "relevant_pmids_reviews": [39114288, 38831203]
    },
    "Q08": {
        "query": "How does gut microbiome dysbiosis contribute to inflammatory bowel disease?",
        "pubmed_search": "gut microbiome dysbiosis inflammatory bowel  mechanism",
        "relevant_pmids": [38992449, 32373209, 40484632],
        "relevant_pmids_reviews": [40065181, 39999800]
    },
    "Q09": {
        "query": "What are the main genetic risk factors for amyotrophic lateral sclerosis?",
        "pubmed_search": "ALS amyotrophic lateral sclerosis genetic risk factors",
        "relevant_pmids": [34873335, 40665049, 35525134],
        "relevant_pmids_reviews": [24126629, 37566027]
    },
    "Q10": {
        "query": "What role does TGF-beta signalling play in cancer immune evasion?",
        "pubmed_search": "TGF-beta signalling cancer immune evasion immunosuppression",
        "relevant_pmids": [30410077, 36096533, 21909397, 39884449],
        "relevant_pmids_reviews": [40701694, 20055717]
    },
    "Q11": {
        "query": "Which biomarkers predict response to trastuzumab in HER2-positive breast cancer?",
        "pubmed_search": "trastuzumab HER2 breast cancer biomarkers treatment response",
        "relevant_pmids": [41260262, 34620206, 39122832],
        "relevant_pmids_reviews": [33934230, 26452383]
    },
    "Q12": {
        "query": "What are the molecular mechanisms of KRAS-driven pancreatic cancer?",
        "pubmed_search": "KRAS pancreatic cancer molecular mechanisms oncogene",
        "relevant_pmids": [38843331, 38843329, 33536616, 38958646],
        "relevant_pmids_reviews": [41484057]
    },
    "Q13": {
        "query": "How do regulatory T cells suppress anti-tumour immunity?",
        "pubmed_search": "regulatory T cells Treg anti-tumour immunity suppression",
        "relevant_pmids": [41448182, 39029466, 36736322],
        "relevant_pmids_reviews": [27995907, 31681327]
    },
    "Q14": {
        "query": "How are stellate cells involved in liver fibrosis?",
        "pubmed_search": "hepatic stellate cells liver fibrosis mechanism",
        "relevant_pmids": [39789010, 35914619, 40213670, 39988062, 39236934],
        "relevant_pmids_reviews": [39063116]
    },
    "Q15": {
        "query": "What are the mechanisms of drug resistance in EGFR-mutant lung cancer?",
        "pubmed_search": "EGFR mutation lung cancer drug resistance mechanisms",
        "relevant_pmids": [32483558, 31911548, 37725242],
        "relevant_pmids_reviews": [31564718, 30404555]
    },
    "Q16": {
        "query": "How does mitochondrial dysfunction contribute to Alzheimer's disease?",
        "pubmed_search": "mitochondrial dysfunction Alzheimer's mechanism",
        "relevant_pmids": [33774476, 36578022, 38164158, 35732922],
        "relevant_pmids_reviews": [40023293, 35522844]
    },
    "Q17": {
        "query": "What is the role of senescent cells in age-related tissue dysfunction?",
        "pubmed_search": "cellular senescence age-related tissue dysfunction",
        "relevant_pmids": [39266768, 40265973, 22048312, 29988130],
        "relevant_pmids_reviews": [39548312]
    },
    "Q18": {
        "query": "Do liver cholangiocytes exhibit cellular plasticity?",
        "pubmed_search": "cholagiocytes liver cell plasticity",
        "relevant_pmids": [38778114, 40480975, 41033615],
        "relevant_pmids_reviews": [40490318, 30382597]
    },
    "Q19": {
        "query": "Which AAV serotypes display tropism to the liver?",
        "pubmed_search": "AAV serotypes tropism liver",
        "relevant_pmids": [39863928, 18414476, 35474733, 37231038],
        "relevant_pmids_reviews": [38543807]
    },
    "Q20": {
        "query": "Do antioxidants enhance cell survival after cryopreservation?",
        "pubmed_search": "antioxidants survival cryopreservation",
        "relevant_pmids": [39585364, 38267808, 40285548, 2480288],
        "relevant_pmids_reviews": [31371631]
    },
}


In [4]:
# ── Validate gold-standard dataset ───────────────────────────────────────────
# Normalise all PMIDs to strings for consistent comparison with PubMed API output
for entry in GOLD_STANDARD.values():
    entry["relevant_pmids"]         = [str(p) for p in entry["relevant_pmids"]]
    entry["relevant_pmids_reviews"] = [str(p) for p in entry["relevant_pmids_reviews"]]

# Summary statistics
rows = []
for qid, entry in GOLD_STANDARD.items():
    n_research = len(entry["relevant_pmids"])
    n_reviews  = len(entry["relevant_pmids_reviews"])
    rows.append({
        "Query ID":         qid,
        "Query":            entry["query"][:55] + "...",
        "Original research": n_research,
        "Reviews":          n_reviews,
        "Total":            n_research + n_reviews,
    })

df_gs = pd.DataFrame(rows).set_index("Query ID")
print(f"\nTotals:")
print(f"  Queries:                   {len(GOLD_STANDARD)}")
print(f"  Original research PMIDs:   {df_gs['Original research'].sum()}")
print(f"  Review PMIDs:              {df_gs['Reviews'].sum()}")
print(f"  Combined PMIDs:            {df_gs['Total'].sum()}")
print(f"  Mean per query (research): {df_gs['Original research'].mean():.1f}")
print(f"  Mean per query (combined): {df_gs['Total'].mean():.1f}")


Totals:
  Queries:                   20
  Original research PMIDs:   71
  Review PMIDs:              33
  Combined PMIDs:            104
  Mean per query (research): 3.5
  Mean per query (combined): 5.2


In [5]:
df_gs

,Query,Original research,Reviews,Total
Query ID,,,,
Q01,What is the mechanistic link between BRCA1 and...,3,2,5
Q02,Which genes are implicated in idiopathic pulmo...,3,2,5
Q03,What are the biomarkers of NASH progression to...,4,1,5
Q04,What mechanisms underlie resistance to anti-PD...,5,1,6
Q05,What is the role of alpha-synuclein aggregatio...,3,2,5
Q06,What drives CAR-T cells to become exhausted?...,3,2,5
Q07,What is the mechanism of action of GIP recepto...,3,2,5
Q08,How does gut microbiome dysbiosis contribute t...,3,2,5
Q09,What are the main genetic risk factors for amy...,3,2,5


## 3. Corpus Construction

For each query, fetch the top 30 PubMed abstracts using the keyword search term (Best Match ranking).
Pool and deduplicate across all queries.
Then explicitly add any gold-standard PMIDs not captured by keyword search.

> **Why Best Match for corpus construction?** The evaluation corpus needs to contain the gold-standard papers. Best Match surfaces highly-cited and relevant papers regardless of date, maximising the chance that manually identified gold-standard PMIDs are present. This differs from the app, which uses `sort=date` to deliver on its recency promise.

In [6]:
def fetch_abstracts_by_pmids(pmids, ncbi_api_key):
    """Fetch abstracts by PMID individually to avoid batch parsing issues."""
    if not pmids:
        return []
    
    papers = []
    for pmid in pmids:
        raw = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params={"db":"pubmed","id":pmid,"rettype":"abstract",
                    "retmode":"text","api_key":ncbi_api_key},
            timeout=30
        ).text
        time.sleep(0.3)

        # Try standard parser first
        parsed = parse_abstracts_with_metadata(raw)
        if parsed:
            papers.extend(parsed)
            continue

        # Fallback parser for alternative format
        pm = re.search(r'PMID:\s*(\d+)', raw)
        if not pm:
            continue

        pubmed_url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        sections = [s.strip() for s in re.split(r'\n\n', raw) if s.strip()]

        title, abstract, yr = "", "", ""
        for i, section in enumerate(sections):
            if i == 0:
                yr_m = re.search(r'\b(19|20)\d{2}\b', section)
                yr = yr_m.group(0) if yr_m else ""
                continue
            if re.match(r'(PMID|PMCID|DOI):', section):
                continue
            if re.search(r'\(\d+\)', section) and len(section.split('\n')) > 1:
                continue
            if not title and len(section) < 300:
                title = section.replace('\n', ' ').strip()
                continue
            if len(section) > len(abstract):
                abstract = section.replace('\n', ' ').strip()

        abstract = clean_abstract(abstract)
        if abstract and len(abstract) > 50:
            papers.append({
                "abstract":   abstract,
                "pmid":       pmid,
                "pubmed_url": pubmed_url,
                "title":      title,
                "authors":    "",
                "year":       yr,
            })

    return papers

def build_eval_corpus(gold_standard, ncbi_api_key, n_per_query=30):
    corpus = {}  # pmid → paper dict

    for qid, entry in gold_standard.items():
        print(f"Fetching corpus for {qid}...")
        raw = fetch_pubmed_abstracts_batched(
            entry["pubmed_search"], ncbi_api_key,
            max_results=n_per_query, batch_size=30
        )
        papers = parse_abstracts_with_metadata(raw)
        for p in papers:
            corpus[p["pmid"]] = p

    # Enrich metadata
    meta = fetch_structured_metadata(list(corpus.keys()), ncbi_api_key)
    for pmid, m in meta.items():
        if pmid in corpus:
            corpus[pmid].update(m)

    # Guarantee gold-standard PMIDs are in corpus
    # Guarantee ALL gold-standard PMIDs are in corpus (research + reviews)
    all_gold_pmids = set()
    for entry in gold_standard.values():
        all_gold_pmids.update(entry["relevant_pmids"])
        all_gold_pmids.update(entry["relevant_pmids_reviews"])

    missing = all_gold_pmids - set(corpus.keys())
    if missing:
        print(f"\nFetching {len(missing)} gold-standard PMIDs not in corpus...")
        extra = fetch_abstracts_by_pmids(list(missing), ncbi_api_key)
        extra_meta = fetch_structured_metadata([p["pmid"] for p in extra], ncbi_api_key)
        for p in extra:
            if p["pmid"] in extra_meta:
                p.update(extra_meta[p["pmid"]])
            corpus[p["pmid"]] = p
        print(f"Added {len(extra)} missing gold-standard abstracts")
    else:
        print("\nAll gold-standard PMIDs present in corpus.")

    papers = list(corpus.values())
    print(f"\nFinal corpus: {len(papers)} unique abstracts")
    return papers

In [7]:
# Only run this once — save corpus to disk for reuse
papers = build_eval_corpus(GOLD_STANDARD, NCBI_API_KEY, n_per_query=60)
with open("eval_corpus.json", "w") as f:
    json.dump(papers, f, indent=2)
print("Corpus saved.")

# Load from disk if already built
with open("eval_corpus.json") as f:
    papers = json.load(f)
print(f"Corpus loaded: {len(papers)} abstracts")

# Verify gold-standard coverage
all_gold = {pmid for v in GOLD_STANDARD.values() for pmid in v["relevant_pmids"]}
corpus_pmids = {p["pmid"] for p in papers}
covered = all_gold & corpus_pmids
print(f"Gold-standard coverage: {len(covered)}/{len(all_gold)} PMIDs in corpus")

Fetching corpus for Q01...
Fetching corpus for Q02...
Fetching corpus for Q03...
Fetching corpus for Q04...
Fetching corpus for Q05...
Fetching corpus for Q06...
Fetching corpus for Q07...
Fetching corpus for Q08...
Fetching corpus for Q09...
Fetching corpus for Q10...
Fetching corpus for Q11...
Fetching corpus for Q12...
Fetching corpus for Q13...
Fetching corpus for Q14...
Fetching corpus for Q15...
Fetching corpus for Q16...
Fetching corpus for Q17...
Fetching corpus for Q18...
Fetching corpus for Q19...
Fetching corpus for Q20...

Fetching 15 gold-standard PMIDs not in corpus...
Added 14 missing gold-standard abstracts

Final corpus: 1049 unique abstracts
Corpus saved.
Corpus loaded: 1049 abstracts
Gold-standard coverage: 70/71 PMIDs in corpus


In [8]:
all_gold = {pmid for v in GOLD_STANDARD.values() 
            for pmid in v["relevant_pmids"] + v["relevant_pmids_reviews"]}
corpus_pmids = {p["pmid"] for p in papers}
missing = all_gold - corpus_pmids
print(f"Remaining missing: {missing}")

Remaining missing: {'39333315'}


## 4. Encoders

In [9]:
ENCODERS = {
    "MiniLM":     "all-MiniLM-L6-v2",
    "PubMedBERT": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract",
    "S-PubMedBert":    "pritamdeka/S-PubMedBert-MS-MARCO",
}

## 5. Retrieval

For each encoder: embed the corpus, store in ChromaDB, retrieve top 10 abstracts per query.

In [10]:
def build_collection(papers, model, collection_name):
    client = chromadb.Client()
    try: client.delete_collection(collection_name)
    except: pass
    col = client.create_collection(collection_name)
    docs_to_embed = [p["abstract"] for p in papers]
    embeddings = model.encode(docs_to_embed, show_progress_bar=True)
    col.add(
        documents=[p["abstract"] for p in papers],
        embeddings=[e.tolist() for e in embeddings],
        metadatas=[{"pmid": p["pmid"], "title": p.get("title", ""),
                    "pubmed_url": p.get("pubmed_url", "")} for p in papers],
        ids=[f"doc_{i}" for i in range(len(papers))]
    )
    return col

def retrieve(query, collection, model, k=10):
    emb = model.encode([query])[0].tolist()
    res = collection.query(
        query_embeddings=[emb], n_results=k,
        include=["documents", "metadatas"]
    )
    return [(meta["pmid"], doc, meta)
            for doc, meta in zip(res["documents"][0], res["metadatas"][0])]

def run_encoder(encoder_name, model_name, papers, gold_standard, k=10):
    print(f"\n{'='*50}")
    print(f"Encoder: {encoder_name} ({model_name})")
    model = SentenceTransformer(model_name)
    col   = build_collection(papers, model, f"eval_{encoder_name}")
    results = {}
    for qid, entry in gold_standard.items():
        if not entry["relevant_pmids"]:
            continue  # skip unannotated queries
        results[qid] = retrieve(entry["query"], col, model, k=k)
    print(f"Retrieval complete for {len(results)} annotated queries")
    return results

In [11]:
# Run all encoders — may take several minutes to download models
all_results = {}
for name, model_name in ENCODERS.items():
    all_results[name] = run_encoder(name, model_name, papers, GOLD_STANDARD, k=10)

print("\nAll encoders complete.")


Encoder: MiniLM (all-MiniLM-L6-v2)


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Retrieval complete for 20 annotated queries

Encoder: PubMedBERT (microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract)


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Retrieval complete for 20 annotated queries

Encoder: S-PubMedBert (pritamdeka/S-PubMedBert-MS-MARCO)


...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Retrieval complete for 20 annotated queries

All encoders complete.


## 6. Metrics

- **Precision@k** — of the top k retrieved abstracts, what fraction are relevant?
- **Recall@k** — of all gold-standard relevant abstracts, what fraction appear in the top k?
- **MRR** — reciprocal rank of the first relevant abstract in the retrieved list (averaged across queries)

In [12]:
def precision_at_k(retrieved_pmids, relevant_pmids, k=5):
    hits = sum(1 for pmid in retrieved_pmids[:k] if pmid in relevant_pmids)
    return hits / k

def recall_at_k(retrieved_pmids, relevant_pmids, k=5):
    if not relevant_pmids:
        return 0
    hits = sum(1 for pmid in retrieved_pmids[:k] if pmid in relevant_pmids)
    return hits / len(relevant_pmids)

def mrr(retrieved_pmids, relevant_pmids):
    for rank, pmid in enumerate(retrieved_pmids, start=1):
        if pmid in relevant_pmids:
            return 1 / rank
    return 0

def score_encoder(encoder_name, results, gold_standard, label_key, k=5):
    """
    label_key: 'relevant_pmids' for original research only,
               'relevant_pmids_combined' for research + reviews
    """
    scores = []
    for qid, retrieved in results.items():
        relevant = set(gold_standard[qid][label_key])
        retrieved_pmids = [pmid for pmid, _, _ in retrieved]
        scores.append({
            "qid":        qid,
            "encoder":    encoder_name,
            "P@5":        precision_at_k(retrieved_pmids, relevant, k=5),
            "R@5":        recall_at_k(retrieved_pmids, relevant, k=5),
            "P@10":       precision_at_k(retrieved_pmids, relevant, k=10),
            "R@10":       recall_at_k(retrieved_pmids, relevant, k=10),
            "MRR":        mrr(retrieved_pmids, relevant),
            "n_relevant": len(relevant),
        })
    return scores

# Add combined PMID list to each entry
for entry in GOLD_STANDARD.values():
    entry["relevant_pmids_combined"] = (
        entry["relevant_pmids"] + entry["relevant_pmids_reviews"]
    )

In [13]:
# ── Evaluation 1: Original research only ─────────────────────────────────────
scores_research = []
for encoder_name, results in all_results.items():
    scores_research.extend(score_encoder(encoder_name, results, GOLD_STANDARD,
                                         label_key="relevant_pmids"))
df_research = pd.DataFrame(scores_research)

# ── Evaluation 2: Combined (research + reviews) ───────────────────────────────
scores_combined = []
for encoder_name, results in all_results.items():
    scores_combined.extend(score_encoder(encoder_name, results, GOLD_STANDARD,
                                         label_key="relevant_pmids_combined"))
df_combined = pd.DataFrame(scores_combined)

print("Scoring complete.")
print(f"  Evaluation 1 (original research): {len(df_research)} rows")
print(f"  Evaluation 2 (combined):          {len(df_combined)} rows")

Scoring complete.
  Evaluation 1 (original research): 60 rows
  Evaluation 2 (combined):          60 rows


## 7. Results

In [14]:
metrics = ["P@5", "R@5", "P@10", "R@10", "MRR"]

# ── Summary tables ────────────────────────────────────────────────────────────
print("Evaluation 1 — Original research PMIDs only")
summary_research = (
    df_research.groupby("encoder")[metrics]
    .mean().round(3)
    .sort_values("MRR", ascending=False)
)
display(summary_research)

print("Evaluation 2 — Combined (research + reviews)")
summary_combined = (
    df_combined.groupby("encoder")[metrics]
    .mean().round(3)
    .sort_values("MRR", ascending=False)
)
display(summary_combined)

# ── Per-query breakdown ───────────────────────────────────────────────────────
for metric in ["MRR", "P@5", "R@5"]:
    print(f"Per-query {metric} — Original research only")
    pivot = df_research.pivot_table(index="qid", columns="encoder", values=metric).round(3)
    pivot.insert(0, "query", [GOLD_STANDARD[qid]["query"][:55]+"..." for qid in pivot.index])
    display(pivot)

Evaluation 1 — Original research PMIDs only


,P@5,R@5,P@10,R@10,MRR
encoder,,,,,
MiniLM,0.07,0.108,0.075,0.218,0.170
S-PubMedBert,0.07,0.108,0.085,0.250,0.165
PubMedBERT,0.04,0.052,0.025,0.068,0.140


Evaluation 2 — Combined (research + reviews)


,P@5,R@5,P@10,R@10,MRR
encoder,,,,,
MiniLM,0.19,0.185,0.165,0.320,0.510
S-PubMedBert,0.20,0.195,0.195,0.382,0.410
PubMedBERT,0.09,0.087,0.055,0.107,0.217


Per-query MRR — Original research only


encoder,query,MiniLM,PubMedBERT,S-PubMedBert
qid,,,,
Q01,What is the mechanistic link between BRCA1 and...,0.000,0.0,0.000
Q02,Which genes are implicated in idiopathic pulmo...,0.125,0.0,0.200
Q03,What are the biomarkers of NASH progression to...,0.125,0.0,0.125
Q04,What mechanisms underlie resistance to anti-PD...,0.125,0.0,0.000
Q05,What is the role of alpha-synuclein aggregatio...,0.000,0.0,0.000
Q06,What drives CAR-T cells to become exhausted?...,0.125,0.1,0.143
Q07,What is the mechanism of action of GIP recepto...,0.000,0.0,0.125
Q08,How does gut microbiome dysbiosis contribute t...,0.000,0.0,0.000
Q09,What are the main genetic risk factors for amy...,0.000,0.0,0.000


Per-query P@5 — Original research only


encoder,query,MiniLM,PubMedBERT,S-PubMedBert
qid,,,,
Q01,What is the mechanistic link between BRCA1 and...,0.0,0.0,0.0
Q02,Which genes are implicated in idiopathic pulmo...,0.0,0.0,0.2
Q03,What are the biomarkers of NASH progression to...,0.0,0.0,0.0
Q04,What mechanisms underlie resistance to anti-PD...,0.0,0.0,0.0
Q05,What is the role of alpha-synuclein aggregatio...,0.0,0.0,0.0
Q06,What drives CAR-T cells to become exhausted?...,0.0,0.0,0.0
Q07,What is the mechanism of action of GIP recepto...,0.0,0.0,0.0
Q08,How does gut microbiome dysbiosis contribute t...,0.0,0.0,0.0
Q09,What are the main genetic risk factors for amy...,0.0,0.0,0.0


Per-query R@5 — Original research only


encoder,query,MiniLM,PubMedBERT,S-PubMedBert
qid,,,,
Q01,What is the mechanistic link between BRCA1 and...,0.000,0.000,0.000
Q02,Which genes are implicated in idiopathic pulmo...,0.000,0.000,0.333
Q03,What are the biomarkers of NASH progression to...,0.000,0.000,0.000
Q04,What mechanisms underlie resistance to anti-PD...,0.000,0.000,0.000
Q05,What is the role of alpha-synuclein aggregatio...,0.000,0.000,0.000
Q06,What drives CAR-T cells to become exhausted?...,0.000,0.000,0.000
Q07,What is the mechanism of action of GIP recepto...,0.000,0.000,0.000
Q08,How does gut microbiome dysbiosis contribute t...,0.000,0.000,0.000
Q09,What are the main genetic risk factors for amy...,0.000,0.000,0.000


## 8. Interpretation

- **Retrieval fine-tuning is the dominant factor, not domain specificity**. PubMedBERT without retrieval fine-tuning underperforms a general model with it (MiniLM).
- **S-PubMedBERT represents the best of both worlds but doesn't dramatically outperform MiniLM** — the MS-MARCO fine-tuning signal is similar for both, and the biomedical vocabulary advantage is modest on this corpus.
- **All encoders perform better on combined (original research articles + reviews) than original research alone**:
  >
  - MiniLM: MRR 0.170 → 0.510 (+200%)
  -  S-PubMedBERT: MRR 0.165 → 0.410 (+148%)
  -  PubMedBERT: MRR 0.140 → 0.217 (+55%)
>
Review abstracts use broad topic-summarising language that aligns well with natural language queries. Original research abstracts use specific technical language that is harder to match semantically regardless of encoder.
>
- **Query specificity matters more than encoder choice**. Queries with distinctive terminology (e.g. cholangiocytes, AAV serotypes, trastuzumab) retrieve well regardless of encoder. Broad mechanistic queries fail regardless of encoder.
- **The primary bottleneck is corpus design, not the encoder**. A topic-specific corpus (as used in the app) would likely produce much higher scores than this mixed 20-topic evaluation corpus.